# Hotel datasets: class balance and rating composition

Exploratory analysis of all `hotel*.csv` files under `hotel-datasets/`:

- Proportion of `class` 0 vs 1 (per file and pooled)
- Which numeric `rating` values appear under each class (counts and row-wise percentages within class)

In [1]:
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
HOTEL_DIR = NOTEBOOK_DIR / "hotel-datasets"
csv_paths = sorted(HOTEL_DIR.glob("hotel*.csv"))
if not csv_paths:
    raise FileNotFoundError(f"No hotel*.csv under {HOTEL_DIR.resolve()}")

print(f"Found {len(csv_paths)} files under {HOTEL_DIR}")

Found 9 files under /var/new_homes/jp/mirko/quantification-over-time/time series qua/hotel-datasets


In [2]:
def load_hotel_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    expected = {"text", "rating", "date", "class"}
    missing = expected - set(df.columns)
    if missing:
        raise ValueError(f"{path.name}: missing columns {missing}")
    df["dataset"] = path.stem
    return df


frames = [load_hotel_csv(p) for p in csv_paths]
all_df = pd.concat(frames, ignore_index=True)
all_df["rating"] = pd.to_numeric(all_df["rating"], errors="coerce")
all_df["class"] = pd.to_numeric(all_df["class"], errors="coerce").astype("Int64")

n_bad_rating = all_df["rating"].isna().sum()
n_bad_class = all_df["class"].isna().sum()
if n_bad_rating or n_bad_class:
    print(f"Warning: NaN after coerce — rating: {n_bad_rating}, class: {n_bad_class}")

all_df.shape

(22128, 5)

## 1. Class proportions (0 vs 1)

Per dataset and over all rows combined.

In [3]:
def class_summary(df: pd.DataFrame, name: str) -> pd.DataFrame:
    vc = df["class"].value_counts(dropna=False).sort_index()
    total = vc.sum()
    prop = (vc / total).rename("proportion")
    out = pd.DataFrame({"count": vc, "proportion": prop})
    out.index.name = "class"
    print(f"\n=== {name} (n={int(total)}) ===")
    display(out)
    return out


per_dataset = {}
for name, g in all_df.groupby("dataset", sort=True):
    per_dataset[name] = class_summary(g, name)

_ = class_summary(all_df, "ALL DATASETS POOLED")


=== hotel1 (n=2529) ===


,count,proportion
class,,
0,1033,0.408462
1,1496,0.591538
<NA>,0,0.0



=== hotel2 (n=5193) ===


,count,proportion
class,,
0,1112,0.214134
1,4081,0.785866
<NA>,0,0.0



=== hotel3 (n=1196) ===


...

,count,proportion
class,,
0,1017,0.507232
1,988,0.492768
<NA>,0,0.0



=== hotel9 (n=2810) ===


,count,proportion
class,,
0,803,0.285765
1,2007,0.714235
<NA>,0,0.0



=== ALL DATASETS POOLED (n=22128) ===


,count,proportion
class,,
0,7958,0.359635
1,14170,0.640365
<NA>,0,0.0


## 2. Ratings that compose each class

Cross-tabulation `class` × `rating`:

- **Count**: how many rows have that pair
- **Within class**: share of that class that has each rating (rows sum to 1 per class)
- **Within rating**: share of that rating that falls in each class (columns sum to 1 per rating)

In [4]:
def rating_by_class_tables(df: pd.DataFrame, title: str):
    sub = df.dropna(subset=["class", "rating"])
    ct = pd.crosstab(sub["class"], sub["rating"], margins=False)
    ct = ct.reindex(sorted(ct.columns), axis=1)
    ct = ct.sort_index()
    row_pct = ct.div(ct.sum(axis=1), axis=0)
    col_pct = ct.div(ct.sum(axis=0), axis=1)
    print(f"\n--- {title} ---")
    print("Counts (class × rating):")
    display(ct)
    print("Row % (within each class, by rating):")
    display(row_pct.round(4))
    print("Column % (within each rating, by class):")
    display(col_pct.round(4))
    return ct, row_pct, col_pct


pooled_ct, pooled_row, pooled_col = rating_by_class_tables(all_df, "ALL DATASETS POOLED")

for name, g in all_df.groupby("dataset", sort=True):
    rating_by_class_tables(g, name)


--- ALL DATASETS POOLED ---
Counts (class × rating):


rating,1,2,3,4,5,6,7,8,9,10
class,,,,,,,,,,
0,82,400,71,360,210,944,693,3244,1954,0
1,0,0,0,0,0,0,0,0,0,14170


Row % (within each class, by rating):


rating,1,2,3,4,5,6,7,8,9,10
class,,,,,,,,,,
0,0.0103,0.0503,0.0089,0.0452,0.0264,0.1186,0.0871,0.4076,0.2455,0.0
1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0


Column % (within each rating, by class):


...

rating,1,2,3,4,5,6,7,8,9,10
class,,,,,,,,,,
0,6,56,7,50,20,117,55,323,169,0
1,0,0,0,0,0,0,0,0,0,2007


Row % (within each class, by rating):


rating,1,2,3,4,5,6,7,8,9,10
class,,,,,,,,,,
0,0.0075,0.0697,0.0087,0.0623,0.0249,0.1457,0.0685,0.4022,0.2105,0.0
1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0


Column % (within each rating, by class):


rating,1,2,3,4,5,6,7,8,9,10
class,,,,,,,,,,
0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


## 3. Quick sanity: unique ratings per class (pooled)

Lists which rating values appear at least once under class 0 vs class 1.

In [5]:
for c in sorted(all_df["class"].dropna().unique()):
    ratings = sorted(all_df.loc[all_df["class"] == c, "rating"].dropna().unique())
    print(f"class {int(c)}: unique ratings = {ratings}")

class 0: unique ratings = [1, 2, 3, 4, 5, 6, 7, 8, 9]
class 1: unique ratings = [10]


## 4. Reviews per chunk period (chunk_report)

For each row in `hotel-datasets-neutral/chunk_report/*_chunks_by_{day|week|month}.csv`: **x** is the timestamp column in that file (**day** → `utc_day`, **week** → `week_start_utc_monday`, **month** → first day of `year_month`), **y** is **`n_reviews`** (review count in that chunk). Nothing else is plotted.

Interactive controls: **Dataset** and **Chunk** (day / week / month). Requires `plotly` and `ipywidgets`.

In [20]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

NOTEBOOK_DIR = Path.cwd()
NEUTRAL_DIR = NOTEBOOK_DIR / "hotel-datasets-neutral"
CHUNK_REPORT_DIR = NEUTRAL_DIR / "chunk_report"

if not CHUNK_REPORT_DIR.is_dir():
    raise FileNotFoundError(
        f"Expected chunk_report under {NEUTRAL_DIR}. Run chunk_report/generate_chunk_reports.py first."
    )

hotels = sorted(p.stem for p in NEUTRAL_DIR.glob("hotel*.csv"))
if not hotels:
    raise FileNotFoundError(f"No hotel*.csv under {NEUTRAL_DIR}")

chunk_tables = {}
for stem in hotels:
    chunk_tables[stem] = {}
    for gran in ("day", "week", "month"):
        rep = CHUNK_REPORT_DIR / f"{stem}_chunks_by_{gran}.csv"
        if not rep.is_file():
            raise FileNotFoundError(rep)
        chunk_tables[stem][gran] = pd.read_csv(rep)


def chunk_period_start_utc(df: pd.DataFrame, gran: str) -> pd.Series:
    """Timestamp column from chunk_report CSVs (one timestamp per chunk row)."""
    if gran == "day":
        return pd.to_datetime(df["utc_day"], utc=True)
    if gran == "week":
        return pd.to_datetime(df["week_start_utc_monday"], utc=True)
    if gran == "month":
        return pd.to_datetime(df["year_month"] + "-01", utc=True)
    raise ValueError(gran)


def chunk_report_figure(hotel: str, gran: str) -> go.Figure:
    """One bar per chunk row: x = timestamp from chunk_report, y = n_reviews for that period."""
    df = chunk_tables[hotel][gran].copy()
    df["_t"] = chunk_period_start_utc(df, gran)
    df = df.sort_values("_t")
    n = len(df)
    fig = go.Figure(
        data=[
            go.Bar(
                x=df["_t"],
                y=df["n_reviews"],
                name="reviews",
                marker_line_width=0,
            )
        ]
    )
    fig.update_layout(
        title=f"{hotel} · {gran} ({n} chunks)",
        xaxis_title="Timestamp (from chunk_report CSV)",
        yaxis_title="Review count in chunk (n_reviews)",
        bargap=0.15,
        template="plotly_white",
        height=480,
    )
    fig.update_xaxes(type="date")
    return fig


dd_hotel = widgets.Dropdown(options=hotels, value=hotels[0], description="Dataset:")
dd_chunk = widgets.Dropdown(
    options=["day", "week", "month"],
    value="day",
    description="Chunk:",
)
out = widgets.Output()


def _redraw(_=None):
    with out:
        out.clear_output(wait=True)
        fig = chunk_report_figure(dd_hotel.value, dd_chunk.value)
        fig.show()


dd_hotel.observe(_redraw, "value")
dd_chunk.observe(_redraw, "value")

display(widgets.HBox([dd_hotel, dd_chunk]), out)
_redraw()

Output()